# Gemma 4 paper: F2P verification of 12 post-cutoff keeps
Runs verify_f2p.py in Kaggle (internet ON). Output: /kaggle/working/keeps_verified.jsonl

In [ ]:
%%writefile /kaggle/working/verify_f2p.py
"""Fail-to-pass / pass-to-pass verifier for Gemma 4 paper keeps.
Run INSIDE the gemma4h sandbox (Kaggle notebook), not on the box.
For each keep: clone repo at base_sha, apply PR test files from merge_sha,
run F2P candidates (expect FAIL), then check out merge_sha src and rerun (expect PASS).
Writes keeps_verified.jsonl with verified=true/false and observed statuses.
Usage: python verify_f2p.py keeps_f2p.jsonl keeps_verified.jsonl [--workdir /kaggle/working/f2p]
"""
import json, subprocess, sys, os, argparse

def sh(cmd, cwd=None, timeout=900):
    p = subprocess.run(cmd, cwd=cwd, shell=True, capture_output=True, text=True, timeout=timeout)
    return p.returncode, p.stdout[-4000:] + p.stderr[-4000:]

def run_tests(repo_dir, tests):
    status = {}
    for t in tests:
        rc, _ = sh(f"python -m pytest -x -q {t!r} -p no:cacheprovider", cwd=repo_dir, timeout=600)
        status[t] = "PASS" if rc == 0 else "FAIL"
    return status

def verify(task, workdir):
    name = task["repo"].replace("/", "__") + f"_{task['pr']}"
    d = os.path.join(workdir, name)
    if not os.path.isdir(d):
        sh(f"git clone --quiet https://github.com/{task['repo']}.git {d}", timeout=1200)
    sh(f"git fetch --quiet origin {task['merge_sha']} {task['base_sha']}", cwd=d)
    sh(f"git checkout --quiet -f {task['base_sha']}", cwd=d)
    sh("python -m pip install -q -e . pytest", cwd=d, timeout=1200)
    tf = " ".join(task["test_files"])
    sh(f"git checkout --quiet {task['merge_sha']} -- {tf}", cwd=d)
    tests = task.get("fail_to_pass_candidates", [])
    before = run_tests(d, tests)
    sf = " ".join(task["src_files"])
    sh(f"git checkout --quiet {task['merge_sha']} -- {sf}", cwd=d)
    after = run_tests(d, tests)
    f2p = [t for t in tests if before[t] == "FAIL" and after[t] == "PASS"]
    rc_full, _ = sh(f"python -m pytest -q {tf} -p no:cacheprovider", cwd=d)
    out = dict(task, before=before, after=after, FAIL_TO_PASS=f2p,
               test_files_green_after=(rc_full == 0), verified=bool(f2p) and rc_full == 0)
    sh("git checkout --quiet -f .", cwd=d)
    return out

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("inp"); ap.add_argument("out")
    ap.add_argument("--workdir", default="/kaggle/working/f2p")
    a = ap.parse_args()
    os.makedirs(a.workdir, exist_ok=True)
    tasks = [json.loads(l) for l in open(a.inp) if l.strip()]
    with open(a.out, "w") as fo:
        for t in tasks:
            try:
                r = verify(t, a.workdir)
            except Exception as e:
                r = dict(t, verified=False, error=str(e)[:500])
            fo.write(json.dumps(r) + "\n"); fo.flush()
            print(t["repo"], t["pr"], "verified" if r.get("verified") else "NOT verified", r.get("FAIL_TO_PASS"))

if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/keeps_f2p.jsonl
{"repo": "Textualize/rich", "pr": 4077, "url": "https://github.com/Textualize/rich/pull/4077", "title": "proxy isatty", "merged_at": "2026-04-12T04:11:19Z", "base_sha": "58ac15120fe254886c10d6253667733c83604cb4", "merge_sha": "19c67b9a3479841e9133bea94607c89ee931d3fc", "src_files": ["rich/file_proxy.py"], "test_files": ["tests/test_file_proxy.py"], "additions": 21, "deletions": 0, "license_check": "MIT", "verified": false, "triage": "keep_bugfix", "triage_reason": "behavior fix, small diff", "fail_to_pass_candidates": ["tests/test_file_proxy.py::test_isatty", "tests/test_file_proxy.py::test_new_lines"]}
{"repo": "Textualize/rich", "pr": 4079, "url": "https://github.com/Textualize/rich/pull/4079", "title": "Inline table code", "merged_at": "2026-04-12T07:45:03Z", "base_sha": "19c67b9a3479841e9133bea94607c89ee931d3fc", "merge_sha": "ffe2edc5968eac19d5493c2d7b27965031a692e9", "src_files": ["rich/markdown.py"], "test_files": ["tests/test_markdown.py"], "additions": 21, "deletions": 3, "license_check": "MIT", "verified": false, "triage": "keep_bugfix", "triage_reason": "behavior fix, small diff", "fail_to_pass_candidates": ["tests/test_markdown.py::test_inline_code_in_table_cells", "tests/test_markdown.py::test_table_with_empty_cells"]}
{"repo": "Textualize/rich", "pr": 4076, "url": "https://github.com/Textualize/rich/pull/4076", "title": "preserve newlines", "merged_at": "2026-04-12T03:59:42Z", "base_sha": "9cb198944f8184df92217efc8b20d3fffa4b4fa0", "merge_sha": "58ac15120fe254886c10d6253667733c83604cb4", "src_files": ["rich/ansi.py"], "test_files": ["tests/test_ansi.py"], "additions": 27, "deletions": 9, "license_check": "MIT", "verified": false, "triage": "keep_bugfix", "triage_reason": "behavior fix, small diff", "fail_to_pass_candidates": ["tests/test_ansi.py::test_decode", "tests/test_ansi.py::test_decode_example", "tests/test_ansi.py::test_decode_newlines", "tests/test_ansi.py::test_strip_private_escape_sequences"]}
{"repo": "Textualize/rich", "pr": 4075, "url": "https://github.com/Textualize/rich/pull/4075", "title": "empty with end", "merged_at": "2026-04-11T09:34:48Z", "base_sha": "1fc7cb210ae52eb099df39818ddb9569e5b14c7b", "merge_sha": "9cb198944f8184df92217efc8b20d3fffa4b4fa0", "src_files": ["rich/console.py"], "test_files": ["tests/test_console.py"], "additions": 31, "deletions": 6, "license_check": "MIT", "verified": false, "triage": "keep_bugfix", "triage_reason": "behavior fix, small diff", "fail_to_pass_candidates": ["tests/test_console.py::test_print", "tests/test_console.py::test_print_empty_with_end"]}
{"repo": "Textualize/rich", "pr": 3934, "url": "https://github.com/Textualize/rich/pull/3934", "title": "empty live", "merged_at": "2026-01-22T17:03:12Z", "base_sha": "6764b24739b580c0c32ff5ef0b971ed4cfcf4f75", "merge_sha": "e9a5680682e0dbaf9e0f3f0fc2a771612307bc3d", "src_files": ["rich/live.py", "rich/live_render.py"], "test_files": ["tests/test_live.py"], "additions": 36, "deletions": 3, "license_check": "MIT", "verified": false, "triage": "keep_bugfix", "triage_reason": "behavior fix, small diff", "fail_to_pass_candidates": ["tests/test_live.py::test_live_empty", "tests/test_live.py::test_live_screen"]}
{"repo": "pallets/click", "pr": 3884, "url": "https://github.com/pallets/click/pull/3884", "title": "Fix `show_envvar` output for a sequence or an empty `envvar`", "merged_at": "2026-10-06T20:59:00Z", "base_sha": "bb695aaecdef95832d167fc1d508921745aa824e", "merge_sha": "8c73ff134480f436faa0d041bd19aaf949aac22c", "src_files": ["src/click/core.py"], "test_files": ["tests/test_options.py"], "additions": 54, "deletions": 10, "license_check": "BSD-3-Clause", "verified": false, "triage": "keep_bugfix", "triage_reason": "behavior fix, small diff", "fail_to_pass_candidates": ["tests/test_options.py::test_missing_envvar", "tests/test_options.py::test_missing_envvar_sequence", "tests/test_options.py::test_show_envvar_empty"]}
{"repo": "pallets/click", "pr": 3860, "url": "https://github.com/pallets/click/pull/3860", "title": "Always list all positional arguments", "merged_at": "2026-09-21T18:31:00Z", "base_sha": "86949d19b930ef68a45566544cb4b9508d895d0d", "merge_sha": "bb695aaecdef95832d167fc1d508921745aa824e", "src_files": ["src/click/core.py"], "test_files": ["tests/test_arguments.py"], "additions": 124, "deletions": 13, "license_check": "BSD-3-Clause", "verified": false, "triage": "keep_bugfix", "triage_reason": "behavior fix, small diff", "fail_to_pass_candidates": ["tests/test_arguments.py::test_argument_get_help_record_never_none", "tests/test_arguments.py::test_argument_help_deprecated_without_help_lists_all", "tests/test_arguments.py::test_argument_help_empty_string_lists_all", "tests/test_arguments.py::test_argument_help_lists_undocumented_arguments", "tests/test_arguments.py::test_argument_help_undocumented_arguments_only_no_section"]}
{"repo": "pallets/click", "pr": 3865, "url": "https://github.com/pallets/click/pull/3865", "title": "Fix short help abbreviation", "merged_at": "2026-09-12T02:35:08Z", "base_sha": "6d5bac5c712195cbfa9666ba0ecbf0573cdea356", "merge_sha": "70689853e39c30e36eb0d83d586cc914a74db9d3", "src_files": ["src/click/utils.py"], "test_files": ["tests/test_utils/test_make_default_short_help.py"], "additions": 49, "deletions": 34, "license_check": "BSD-3-Clause", "verified": false, "triage": "keep_bugfix", "triage_reason": "behavior fix, small diff", "fail_to_pass_candidates": ["tests/test_utils/test_make_default_short_help.py::test_make_default_short_help"], "f2p_note": "parametrized; new ids e.g. 'dot ends text','lowercase after dot','abbreviation inside sentence','negative width' (manual, Oct 28 11 AM IST)"}
{"repo": "pallets/flask", "pr": 6096, "url": "https://github.com/pallets/flask/pull/6096", "title": "Fix `.partition(\":\")` usage on potential IPv6 addresses", "merged_at": "2026-08-11T19:48:41Z", "base_sha": "514fc6b3e8402e4c646d5284e97a4f0ab50a7c4b", "merge_sha": "05e9c6bd630ecf4ec0ec884b1fc7901663737bc7", "src_files": ["src/flask/app.py", "src/flask/testing.py"], "test_files": ["tests/test_basic.py", "tests/test_testing.py"], "additions": 23, "deletions": 5, "license_check": "BSD-3-Clause", "verified": false, "triage": "keep_bugfix", "triage_reason": "behavior fix, small diff", "fail_to_pass_candidates": ["tests/test_testing.py::test_session_transaction_ipv6"]}
{"repo": "pallets/flask", "pr": 5928, "url": "https://github.com/pallets/flask/pull/5928", "title": "all teardown callbacks are called despite errors", "merged_at": "2026-02-20T04:00:34Z", "base_sha": "7b0088693ece1bd3a9238a6fdf56ed8df7a4d43b", "merge_sha": "c34d6e81fd8e405e6d4178bf24b364918811ef17", "src_files": ["src/flask/app.py", "src/flask/ctx.py", "src/flask/helpers.py"], "test_files": ["tests/test_appctx.py", "tests/test_basic.py", "tests/test_blueprints.py", "tests/test_helpers.py", "tests/test_testing.py"], "additions": 194, "deletions": 80, "license_check": "BSD-3-Clause", "verified": false, "triage": "keep_bugfix", "triage_reason": "behavior fix, small diff", "fail_to_pass_candidates": ["tests/test_appctx.py::test_robust_teardown", "tests/test_basic.py::test_static_url_path", "tests/test_blueprints.py::test_templates_and_static"]}
{"repo": "pallets/flask", "pr": 5917, "url": "https://github.com/pallets/flask/pull/5917", "title": "fix provide_automatic_options override", "merged_at": "2026-02-12T21:07:51Z", "base_sha": "d3b78fd18a8d9e224cb9ef58a23cec9b1ffc9ce9", "merge_sha": "12e95c93b488725f80753f34b2e0d24838ca4646", "src_files": ["src/flask/sansio/app.py"], "test_files": ["tests/test_basic.py", "tests/test_blueprints.py", "tests/test_cli.py", "tests/test_views.py"], "additions": 96, "deletions": 86, "license_check": "BSD-3-Clause", "verified": false, "triage": "keep_bugfix", "triage_reason": "behavior fix, small diff", "fail_to_pass_candidates": ["tests/test_basic.py::test_method_route_no_methods", "tests/test_basic.py::test_provide_automatic_options_arg_disable", "tests/test_basic.py::test_provide_automatic_options_attr_disable", "tests/test_basic.py::test_provide_automatic_options_attr_enable", "tests/test_basic.py::test_provide_automatic_options_method_disable", "tests/test_blueprints.py::test_default_static_max_age", "tests/test_blueprints.py::test_templates_and_static", "tests/test_cli.py::test_sort", "tests/test_views.py::test_provide_automatic_options_method_disable", "tests/test_views.py::test_view_provide_automatic_options_attr_disable", "tests/test_views.py::test_view_provide_automatic_options_attr_enable"]}
{"repo": "pallets/werkzeug", "pr": 3298, "url": "https://github.com/pallets/werkzeug/pull/3298", "title": "don't modify SERVER_NAME or SERVER_PORT", "merged_at": "2026-09-25T04:30:12Z", "base_sha": "25663ddc295879d362a206b4c33f16ea2e694392", "merge_sha": "fa93b5cee027bc9baadba67f21360a8a39b2317b", "src_files": ["src/werkzeug/middleware/proxy_fix.py"], "test_files": ["tests/middleware/test_proxy_fix.py"], "additions": 38, "deletions": 38, "license_check": "BSD-3-Clause", "verified": false, "triage": "keep_bugfix", "triage_reason": "behavior fix, small diff", "fail_to_pass_candidates": ["tests/middleware/test_proxy_fix.py::test_proxy_fix"], "f2p_note": "parametrized; changed case id 'port, no host' expects http://spam:9000/ (manual, Oct 28 11 AM IST)"}


In [ ]:
!cd /kaggle/working && python verify_f2p.py keeps_f2p.jsonl keeps_verified.jsonl --workdir /kaggle/working/f2p

In [ ]:
import json
for l in open('/kaggle/working/keeps_verified.jsonl'):
    t=json.loads(l); print(t['repo'],t['pr'],t['verified'],t['FAIL_TO_PASS'])